# Encontro 3 · Hands-on: chunking no caso da sua equipe

## O entregável

Um arquivo **`encontro-03/NOTA-CHUNKING.md`** no fork da equipe, com no máximo uma página, a partir do modelo que já está na pasta. Ele tem quatro partes:

1. a estratégia de chunking e de busca adotada;
2. **por quê**, com a tabela do placar que este notebook gera;
3. o que a estratégia **não** resolve, com um caso que continua falhando;
4. quais metadados a ingestão precisa garantir para a estratégia funcionar.

**Está pronto quando:** a tabela tem pelo menos três configurações comparadas, a nota cita os números dela e o arquivo está no fork.

## Como usar

São quatro passos numerados. Cada passo diz no topo o que produz. Se travar, cada passo tem uma célula **Travou?** que deixa tudo no ponto certo para você seguir.

As caixas *Para ir além* são para quem terminou o entregável. Primeiro a nota, depois as caixas.

Se perguntarem "em que passo você está?", responda com o número.

## Passo 1 · Escolher o corpus e medir a linha de base

**Produz:** o placar da configuração mais simples, aquela que todo tutorial usa.

Como nos outros encontros, **o hands-on é na wiki da Aurora para a turma toda**, inclusive para as equipes que trocaram de domínio: é o gabarito que permite comparar os grupos. Não mude nada na célula abaixo.

**Equipe que trocou de domínio:** no bloco de equipe, rode este mesmo notebook sobre o corpus do `projeto/`. Use `meu-corpus-exemplo/` e `meus-casos-exemplo.json` como modelo do formato e troque as duas linhas marcadas. A nota de chunking da equipe é sobre o domínio de vocês.

In [ ]:
import rag_aurora as r

PASTA_CORPUS = r.CORPUS        # domínio próprio: PASTA_CORPUS = "../projeto/corpus"
ARQUIVO_CASOS = r.CASOS        # domínio próprio: ARQUIVO_CASOS = "../projeto/casos-busca.json"

docs = r.carregar_corpus(PASTA_CORPUS)
casos = r.carregar_casos(ARQUIVO_CASOS)
emb = r.Embeddings()
print(len(docs), "páginas ·", len(casos), "casos")

In [ ]:
experimentos = []
base = r.montar_buscador(docs, emb, estrategia="tamanho", tamanho=400, busca="vetor")
r.registrar(experimentos, "base: tamanho 400 · vetor", r.avaliar(base, casos, k=3))
r.comparar(experimentos)

## Passo 2 · Mudar uma coisa de cada vez

**Produz:** pelo menos três linhas novas na tabela.

Edite os parâmetros, troque o `NOME` e rode a célula. Rode de novo para cada configuração. Mude **uma coisa por vez**: se mudar três, não vai saber qual delas fez diferença.

| Parâmetro | Opções |
|---|---|
| `ESTRATEGIA` | `"tamanho"` ou `"estrutura"` |
| `TAMANHO`, `SOBREPOSICAO` | só valem para `"tamanho"`: por exemplo 400/0, 800/100 |
| `PREFIXO` | só vale para `"estrutura"`: escrever "Título > Seção" no chunk |
| `BUSCA` | `"vetor"`, `"bm25"` ou `"hibrida"` |
| `SO_VIGENTE` | para cada política, só a página mais recente |
| `SO_OFICIAL` | descarta páginas que não são oficiais |

Uma sequência que costuma render: trocar a estratégia; depois a busca; depois ligar `SO_VIGENTE`; depois `SO_OFICIAL`.

In [ ]:
# Travou? Descomente as linhas abaixo e rode. Elas refazem o passo 1 com a wiki da Aurora.
# import rag_aurora as r
# docs, casos, emb = r.carregar_corpus(), r.carregar_casos(), r.Embeddings()
# experimentos = []
# r.registrar(experimentos, "base: tamanho 400 · vetor", r.avaliar(r.montar_buscador(docs, emb, "tamanho"), casos))

In [ ]:
NOME         = "estrutura · vetor"
ESTRATEGIA   = "estrutura"
TAMANHO      = 400
SOBREPOSICAO = 0
PREFIXO      = True
BUSCA        = "vetor"
SO_VIGENTE   = False
SO_OFICIAL   = False

buscador = r.montar_buscador(docs, emb, ESTRATEGIA, TAMANHO, SOBREPOSICAO, PREFIXO, BUSCA, SO_VIGENTE, SO_OFICIAL)
r.registrar(experimentos, NOME, r.avaliar(buscador, casos, k=3))
r.comparar(experimentos)

> **Para ir além (opcional).** Troque `k=3` por `k=1` e por `k=5` na melhor configuração. Com `k` maior, "achou" sobe e "contaminou" também. Qual dos dois custa mais caro para um agente: não ter o trecho certo, ou ter o trecho certo junto com um errado?

## Passo 3 · Olhar o que ainda falha

**Produz:** um caso que continua falhando, com a causa diagnosticada. Ele vai para a parte 3 da nota.

Escolha a melhor configuração da tabela e reconstrua o buscador com ela na célula abaixo.

In [ ]:
# Travou? Esta é uma configuração boa conhecida para a wiki da Aurora. Use-a se precisar.
# buscador = r.montar_buscador(docs, emb, "estrutura", busca="hibrida", so_vigente=True, so_oficial=True)

In [ ]:
linhas = r.avaliar(buscador, casos, k=3)
print("Casos que falharam:", r.falhas(linhas) or "nenhum")
print("Perguntas sem resposta (o placar não mede):", [c["id"] for c in casos if c["categoria"] == "sem-resposta"])

In [ ]:
CASO = "S1"     # troque por um dos ids acima
caso = next(c for c in casos if c["id"] == CASO)
print(caso["pergunta"])
print("Esperado:", caso.get("resposta_esperada"), "\n")
r.mostrar(buscador(caso["pergunta"], 3))

Classifique a causa do caso que você escolheu:

- **chunking:** o trecho certo existe, mas foi cortado ou misturado com outro assunto;
- **corpus:** a informação não existe na base, ou está desatualizada;
- **conflito:** duas fontes respondem coisas diferentes, e o retriever não tem como saber qual vale;
- **termo exato:** a resposta depende de um código ou número que a busca por sentido não distingue;
- **sem resposta:** a busca trouxe *alguma coisa* para uma pergunta que a base não responde.

Se a causa for "sem resposta", nenhuma configuração de busca resolve. Escreva na nota onde o "não sei" deveria ser decidido.

> **Para ir além (opcional).** Um *reranker* relê os candidatos junto com a pergunta e reordena. Teste com `from sentence_transformers import CrossEncoder` e o modelo `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`: busque 10 candidatos, pontue cada par (pergunta, chunk) com `.predict`, fique com os 3 melhores. Ele resolve os casos de termo exato? E o de versão velha?

## Passo 4 · Escrever a nota

**Produz:** o entregável.

A célula abaixo gera a tabela em markdown. Cole-a na parte 2 do `NOTA-CHUNKING.md` e preencha as outras partes. Depois, faça o commit no fork da equipe.

In [ ]:
print(r.tabela_markdown(experimentos))

**Confira antes de entregar:**

- [ ] a tabela tem a linha de base e pelo menos três outras configurações;
- [ ] a parte 2 cita números da tabela, e não só "ficou melhor";
- [ ] a parte 3 tem um caso real que continua falhando, com a causa;
- [ ] a parte 4 diz quem preenche cada metadado e o que acontece quando ele está errado;
- [ ] o arquivo está em `encontro-03/NOTA-CHUNKING.md` no fork.

> **Para ir além (opcional).** Rode o placar com outro modelo de embeddings, por exemplo `r.Embeddings("intfloat/multilingual-e5-base")` (cerca de 1 GB). O ganho compensa o tamanho e a latência? Leve a resposta para a discussão de custo do Encontro 5.